# Bootstrapping Full
Esta sección repite lo visto en el modelado con la diferencia que utiliza todo el dataset y no aplica train/test con fines de analizar posibles diferencias y efecto que podría ocasionar la partición del dataset

# 1. LR Full

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_lr_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

LR_PARAMS = dict(
    solver="liblinear",
    C=1.0,
    max_iter=2000,
    random_state=BASE_SEED,
    class_weight="balanced"
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

# Usar TODO el dataset sin reset_index
X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 LR FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "lr_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"lr_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "lr_full_detalle.csv"
resumen_path = CARPETA_OUT / "lr_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("LR FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 LR FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 2. RF Full

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_rf_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

RF_PARAMS = dict(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=BASE_SEED,
    n_jobs=-1
)

MODELO = RandomForestClassifier(**RF_PARAMS)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\nRF FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "rf_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"rf_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "rf_full_detalle.csv"
resumen_path = CARPETA_OUT / "rf_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("RF FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌲 RF FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌲 RF FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌲 RF FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 3. DT Full

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_dt_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

DT_PARAMS = dict(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=BASE_SEED
)

MODELO = DecisionTreeClassifier(**DT_PARAMS)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\nDT FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "dt_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"dt_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "dt_full_detalle.csv"
resumen_path = CARPETA_OUT / "dt_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("DT FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌳 DT FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌳 DT FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌳 DT FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500


# 4. KNN Full

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_knn_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

KNN_PARAMS = dict(
    n_neighbors=5,
    weights="distance",
    metric="euclidean",
    n_jobs=-1
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(**KNN_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

# Usar TODO el dataset sin reset_index
X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\nKNN FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "knn_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"knn_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "knn_full_detalle.csv"
resumen_path = CARPETA_OUT / "knn_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1",
    "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("KNN FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 KNN FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 KNN FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 KNN FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/5

# Extra. SVM Full

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.svm import SVC

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_svm_full_dataset")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

SVM_PARAMS = dict(
    C=1.0,
    kernel="rbf",
    gamma="scale",
    class_weight="balanced",
    random_state=BASE_SEED
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", SVC(**SVM_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_eval)
            roc = roc_auc_score(y_eval, y_score)
            pr = average_precision_score(y_eval, y_score)
        elif hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

X_train_all = df.drop(columns=cols_signo).to_numpy(copy=False)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\nSVM FULL -> {signo}")

    y_train_all = df[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "svm_full",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"svm_full_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "svm_full_detalle.csv"
resumen_path = CARPETA_OUT / "svm_full_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("SVM FULL terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🌀 SVM FULL -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌀 SVM FULL -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🌀 SVM FULL -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/5

# Resultados

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# =====================================================
# RUTAS DE LOS RESÚMENES
# =====================================================
BASE_PATH = Path("../4_results")

resumenes = {
    "Random Forest": BASE_PATH / "modelo_rf_full_dataset" / "rf_full_resumen.csv",
    "Árbol de Decisión": BASE_PATH / "modelo_dt_full_dataset" / "dt_full_resumen.csv",
    "KNN": BASE_PATH / "modelo_knn_full_dataset" / "knn_full_resumen.csv",
    "Regresión Logística": BASE_PATH / "modelo_lr_full_dataset" / "lr_full_resumen.csv",
    "SVM": BASE_PATH / "modelo_svm_full_dataset" / "svm_full_resumen.csv",
}

medias_modelos = {}

# =====================================================
# CARGA Y PROCESAMIENTO
# =====================================================
for nombre_modelo, ruta in resumenes.items():
    print(f"\n {nombre_modelo}")
    print("-" * 50)

    if not ruta.exists():
        print(f" No se encontró el archivo: {ruta}")
        continue

    df_resumen = pd.read_csv(ruta, header=[0, 1], index_col=0)

    # Extraer solo las medias
    df_medias = df_resumen.xs("mean", axis=1, level=1)

    # Redondear
    df_medias = df_medias.round(3)

    medias_modelos[nombre_modelo] = df_medias

    display(df_medias.style.format("{:.3f}"))



📊 Random Forest
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_tp_mean,cv_fp_mean,cv_tn_mean,cv_fn_mean
signo,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.502,0.502,0.494,0.489,0.502,0.567,4.937,4.890,5.110,5.063
signo_zodiacal_aries,0.500,0.498,0.498,0.486,0.484,0.499,0.566,4.862,4.899,5.101,5.138
signo_zodiacal_cancer,0.500,0.496,0.496,0.485,0.482,0.497,0.564,4.848,4.921,5.079,5.152
signo_zodiacal_capricornio,0.500,0.500,0.500,0.488,0.485,0.501,0.567,4.880,4.875,5.125,5.120
signo_zodiacal_escorpio,0.500,0.500,0.499,0.489,0.486,0.502,0.568,4.892,4.892,5.108,5.108
signo_zodiacal_geminis,0.500,0.498,0.499,0.482,0.482,0.498,0.565,4.825,4.861,5.139,5.175
signo_zodiacal_leo,0.500,0.496,0.495,0.484,0.481,0.496,0.564,4.838,4.918,5.082,5.162
signo_zodiacal_libra,0.500,0.503,0.503,0.489,0.487,0.505,0.570,4.894,4.830,5.170,5.106
signo_zodiacal_piscis,0.500,0.500,0.500,0.488,0.486,0.500,0.564,4.882,4.877,5.123,5.118



📊 Árbol de Decisión
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_tp_mean,cv_fp_mean,cv_tn_mean,cv_fn_mean
signo,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.502,0.502,0.503,0.494,0.502,0.514,5.028,4.989,5.011,4.972
signo_zodiacal_aries,0.500,0.503,0.504,0.500,0.494,0.503,0.515,5.001,4.934,5.066,4.999
signo_zodiacal_cancer,0.500,0.500,0.500,0.502,0.493,0.500,0.513,5.018,5.014,4.986,4.982
signo_zodiacal_capricornio,0.500,0.499,0.498,0.495,0.488,0.499,0.513,4.954,4.972,5.028,5.046
signo_zodiacal_escorpio,0.500,0.503,0.503,0.496,0.491,0.503,0.515,4.961,4.903,5.097,5.039
signo_zodiacal_geminis,0.500,0.501,0.501,0.496,0.490,0.501,0.513,4.964,4.944,5.056,5.036
signo_zodiacal_leo,0.500,0.499,0.499,0.497,0.490,0.499,0.513,4.974,5.003,4.997,5.026
signo_zodiacal_libra,0.500,0.503,0.503,0.500,0.493,0.503,0.514,5.002,4.944,5.056,4.998
signo_zodiacal_piscis,0.500,0.502,0.502,0.502,0.494,0.502,0.514,5.022,4.984,5.016,4.978



📊 KNN
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_tp_mean,cv_fp_mean,cv_tn_mean,cv_fn_mean
signo,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.498,0.497,0.505,0.474,0.496,0.563,5.053,5.084,4.916,4.947
signo_zodiacal_aries,0.500,0.500,0.498,0.514,0.477,0.498,0.564,5.138,5.141,4.859,4.862
signo_zodiacal_cancer,0.500,0.498,0.495,0.497,0.468,0.499,0.565,4.972,5.006,4.994,5.028
signo_zodiacal_capricornio,0.500,0.503,0.498,0.499,0.468,0.499,0.566,4.988,4.934,5.066,5.012
signo_zodiacal_escorpio,0.500,0.500,0.497,0.503,0.471,0.500,0.566,5.034,5.026,4.974,4.966
signo_zodiacal_geminis,0.500,0.496,0.493,0.490,0.462,0.497,0.564,4.896,4.975,5.025,5.104
signo_zodiacal_leo,0.500,0.497,0.496,0.502,0.469,0.497,0.564,5.022,5.087,4.913,4.978
signo_zodiacal_libra,0.500,0.501,0.498,0.493,0.468,0.502,0.566,4.931,4.913,5.087,5.069
signo_zodiacal_piscis,0.500,0.500,0.495,0.495,0.468,0.503,0.568,4.945,4.947,5.053,5.055



📊 Regresión Logística
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_tp_mean,cv_fp_mean,cv_tn_mean,cv_fn_mean
signo,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.502,0.503,0.505,0.496,0.501,0.569,5.054,5.004,4.996,4.946
signo_zodiacal_aries,0.500,0.501,0.500,0.500,0.493,0.501,0.569,5.004,4.985,5.015,4.996
signo_zodiacal_cancer,0.500,0.496,0.497,0.498,0.490,0.497,0.567,4.976,5.048,4.952,5.024
signo_zodiacal_capricornio,0.500,0.501,0.502,0.499,0.492,0.501,0.568,4.990,4.973,5.027,5.010
signo_zodiacal_escorpio,0.500,0.497,0.497,0.496,0.489,0.497,0.567,4.962,5.029,4.971,5.038
signo_zodiacal_geminis,0.500,0.503,0.505,0.502,0.495,0.501,0.569,5.019,4.967,5.033,4.981
signo_zodiacal_leo,0.500,0.497,0.497,0.495,0.489,0.498,0.567,4.952,5.020,4.980,5.048
signo_zodiacal_libra,0.500,0.501,0.500,0.500,0.492,0.499,0.568,4.998,4.984,5.016,5.002
signo_zodiacal_piscis,0.500,0.499,0.500,0.502,0.494,0.498,0.565,5.018,5.036,4.964,4.982



📊 SVM
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_tp_mean,cv_fp_mean,cv_tn_mean,cv_fn_mean
signo,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.502,0.503,0.508,0.493,0.501,0.569,5.076,5.037,4.963,4.924
signo_zodiacal_aries,0.500,0.499,0.498,0.506,0.490,0.497,0.567,5.064,5.084,4.916,4.936
signo_zodiacal_cancer,0.500,0.497,0.497,0.503,0.488,0.498,0.565,5.029,5.085,4.915,4.971
signo_zodiacal_capricornio,0.500,0.499,0.500,0.489,0.482,0.503,0.570,4.894,4.905,5.095,5.106
signo_zodiacal_escorpio,0.500,0.497,0.497,0.488,0.480,0.498,0.569,4.878,4.935,5.065,5.122
signo_zodiacal_geminis,0.500,0.501,0.503,0.502,0.490,0.500,0.568,5.024,5.014,4.986,4.976
signo_zodiacal_leo,0.500,0.496,0.496,0.494,0.483,0.495,0.567,4.940,5.022,4.978,5.060
signo_zodiacal_libra,0.500,0.499,0.498,0.488,0.481,0.500,0.568,4.880,4.900,5.100,5.120
signo_zodiacal_piscis,0.500,0.497,0.497,0.502,0.488,0.497,0.565,5.024,5.076,4.924,4.976
